## Problema del negocio

¿Que tan agotado está el colaborador?: Regresión

¿Está en riesgo de burnout?: SI o NO (0 o 1)

# ¿Qué es el ML?

Supervisado vs No Supervisado


-Supervisado: El modelo aprende comparando su respuesta con real y corrige sus errores.

-No supervisado: Exporación sin mapa. El modelo descubre patrones por sí solo

In [1]:
import pandas as pd
import numpy as np

#Machine Learning (scikit-lean)
from sklearn.linear_model import LinearRegression, LogisticRegression
#LinearRegresion -> MOdelo predice numero continuos
#LogisticRegresion -> Modelo que predice categorias (0 =  sin burnout, 1 = burnout)

from sklearn.model_selection import train_test_split
#Devidir los datos en dos grupos separados:
# train: el modelo aprende aprende de estos datos
# test: el modelo es evaluado con estos datos que nunca vió.

from sklearn.preprocessing import OneHotEncoder
#Convertir columnas en texto

from sklearn.metrics import (
    mean_absolute_error, #MAE -> error promedio en predicciones
    mean_squared_error, #MSE -> Error promedio prenalizando errores grandes.
    r2_score,  # que % de la variación explica el modelo (0 a 1).
    accuracy_score, # Tabla de aciertos y tipos de errores.
    classification_report # Reporte con precision, recall y F1 por clase.
)


In [2]:
df = pd.read_csv("talentwell_data.csv")

In [3]:
df.head()

,horas_trabajo_dia,modalidad,horas_sueno,reuniones_dia,anios_experiencia,actividad_fisica_semanal,score_agotamiento,riesgo_burnout
0,9.5,remoto,7.1,4,13.6,0,4.80,0
1,8.2,hibrido,4.8,3,11.6,0,5.30,0
2,9.8,presencial,6.9,2,1.6,0,5.19,1
3,11.5,remoto,4.0,3,3.4,3,7.97,1
4,8.0,remoto,7.3,4,9.6,1,4.79,0


In [4]:
filas, columnas = df.shape
print('El dataset tiene {} colaboradores y {} variables.'.format(filas, columnas))

El dataset tiene 1500 colaboradores y 8 variables.


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   horas_trabajo_dia         1500 non-null   float64
 1   modalidad                 1500 non-null   str    
 2   horas_sueno               1500 non-null   float64
 3   reuniones_dia             1500 non-null   int64  
 4   anios_experiencia         1500 non-null   float64
 5   actividad_fisica_semanal  1500 non-null   int64  
 6   score_agotamiento         1500 non-null   float64
 7   riesgo_burnout            1500 non-null   int64  
dtypes: float64(4), int64(3), str(1)
memory usage: 105.1 KB


In [6]:
df.describe().round(2)

,horas_trabajo_dia,horas_sueno,reuniones_dia,anios_experiencia,actividad_fisica_semanal,score_agotamiento,riesgo_burnout
count,1500.00,1500.00,1500.00,1500.00,1500.00,1500.00,1500.00
mean,8.61,6.46,3.93,4.00,2.19,4.98,0.23
std,1.96,1.22,1.96,4.02,1.47,1.61,0.42
min,4.00,3.00,0.00,0.00,0.00,0.00,0.00
25%,7.30,5.60,3.00,1.10,1.00,3.84,0.00
50%,8.60,6.40,4.00,2.70,2.00,5.02,0.00
75%,9.90,7.20,5.00,5.60,3.00,6.08,0.00
max,16.00,10.00,11.00,25.00,5.00,10.00,1.00


# Preparación de los datos

In [7]:
#OneHotEncoding ( OHE )

ohe = OneHotEncoder(

    sparse_output=False, 

    drop='first'
)

#Aprender que categorias tenemos, admeás convierte cada fila en columnas de 0 y 1
modalidad_encoded = ohe.fit_transform(df[['modalidad']])

nombres_columnas = ohe.get_feature_names_out(['modalidad']).tolist()
print("Columnas creadas por OHE: ", nombres_columnas)

Columnas creadas por OHE:  ['modalidad_presencial', 'modalidad_remoto']


In [8]:
#Agregar ñlas columnas nuevas al df orginal
for i, col in enumerate(nombres_columnas):
#i = 0, col= modalidad pres
    df[col] = modalidad_encoded[:,i] #Seleccionar todas las filas de la columna i.

print("Valores unicos encoding: " )
print(df[['modalidad'] + nombres_columnas].drop_duplicates().sort_values('modalidad'))


Valores unicos encoding: 
    modalidad  modalidad_presencial  modalidad_remoto
1     hibrido                   0.0               0.0
2  presencial                   1.0               0.0
0      remoto                   0.0               1.0


# Definir Features (X) y Target (y)

En ML siempre separamos dos cosas:

X -> Lo que el modelo recibre para predrecir

y -> Lo que el modelo produce o entrega como prediccion

In [9]:
features = [
    'horas_trabajo_dia',
    'horas_sueno',
    'reuniones_dia',
    'actividad_fisica_semanal',
    'anios_experiencia',
    'modalidad_presencial',
    'modalidad_remoto'
]


X = df[features]
y_reg = df['score_agotamiento']
y_cls = df['riesgo_burnout']

# Regresion Lineal: Predecir el score de agotamiento

¿Train y test?

-Overfitting ( Sobreajuste )

Solucion: 80% de los datos para entrenar y 20% para evaluar ( DATOS QUE NO VIÓ )

In [10]:
# Dividir en conjuntos entrenaminetos y prueba

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_reg,
    test_size=0.2, #20 de los datos van al conjutnos de prueba
    
    random_state=42 #Semilla aletoria
    
)


print('Entrenamiento: {} colaboradores -> el modelo Aprende de estos.'.format(X_train.shape[0]))
print(' Preuba: {} colaboradores -> el modelo es Evaluado con estos'. format(X_test.shape[0]))

Entrenamiento: 1200 colaboradores -> el modelo Aprende de estos.
 Preuba: 300 colaboradores -> el modelo es Evaluado con estos


In [11]:
#Crear y entrenar el modelo de regresión lineal

modelo_lineal = LinearRegression()

modelo_lineal.fit(X_train, y_train)

print("Modelo entrenado correctamente")


Modelo entrenado correctamente


In [14]:
#Los coeficientes

coeficientes = pd.DataFrame({
    'variable': features,
    'coeficiente': modelo_lineal.coef_
}
)

coeficientes = coeficientes.sort_values('coeficiente', ascending=False)

print("Lo que el modelo aprendió: ")

print(coeficientes.to_string(index=False))

Lo que el modelo aprendió: 
                variable  coeficiente
       horas_trabajo_dia     0.472838
        modalidad_remoto     0.380606
           reuniones_dia     0.354788
       anios_experiencia     0.005304
actividad_fisica_semanal    -0.232357
    modalidad_presencial    -0.360631
             horas_sueno    -0.484095


In [16]:
#Hacer prediccion
y_pred_reg = modelo_lineal.predict(X_test)

#Comparar predicciones vs vcalores reales para los primeros 8 casos
comparacion = pd.DataFrame({
    'Score real': y_test.values[:8].round(2),
    'score predicho': y_pred_reg[:8].round(2),
    'Error': (y_test.values[:8]- y_pred_reg[:8]).round(2)
})

print('Real vs predicho (primeros 8 colaboradores del conjunto de prueba):')
print()
print(comparacion.to_string(index=False))

Real vs predicho (primeros 8 colaboradores del conjunto de prueba):

 Score real  score predicho  Error
       7.10            7.07   0.03
       4.72            5.33  -0.61
       5.31            4.24   1.07
       4.81            5.25  -0.44
       3.29            4.38  -1.09
       3.70            3.30   0.40
       2.44            2.18   0.26
       5.32            5.26   0.06


In [ ]:
#Evaluar las metriocas para el modelo de regresión.
#r2
#mae
#rmse